# Nairobi OS: High-Performance Benchmarking

This notebook demonstrates the extreme resource efficiency of Nairobi OS compared to standard data science tools (Pandas). 

**Author**: Kevin Chege  
**Location**: Nairobi  
**License**: PolyForm Noncommercial License 1.0.0

## 1. Install Nairobi OS
We will install the latest version from PyPI.

In [ ]:
!pip install nairobi-os

## 2. Initialize Refinery with D-Bus Bootstrap
Kaggle and Colab containers do not have a D-Bus session running by default. This cell installs D-Bus infrastructure, launches a D-Bus session, and ignites the Heavy Iron refinery.

In [ ]:
import os
import subprocess
from pathlib import Path
import nairobi_os

print("🛠 1. Installing D-Bus Infrastructure...")
subprocess.run(["apt-get", "update"], capture_output=True)
subprocess.run(["apt-get", "install", "-y", "dbus-x11"], capture_output=True)

print("🔌 2. Initializing D-Bus Session...")
dbus_out = subprocess.check_output(["dbus-launch"]).decode()
for line in dbus_out.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        # Inject the D-Bus address into the Python environment
        os.environ[k] = v.replace(";", "").replace("'", "").replace('"', '')

print("🔐 3. Granting Executable Permissions...")
bin_path = Path(nairobi_os.__file__).parent / "bin" / "nairobi-axum-refinery"
os.chmod(bin_path, 0o755)

print("🔥 4. Igniting the Heavy Iron...")
try:
    nairobi_os.start_refinery()
    print("✅ EMPIRE ONLINE")
except Exception as e:
    print(f"\n💥 Ignition Failed. Dumping Forensic Logs:")
    os.system("cat ~/.nairobi_refinery.log")

## 3. Generate Benchmark Dataset
We'll generate a synthetic dataset with 5 million rows (~400MB) to test memory efficiency.

In [ ]:
import pandas as pd
import numpy as np

num_rows = 5_000_000
df = pd.DataFrame({
    'target': np.random.randn(num_rows),
    'col1': np.random.randn(num_rows),
    'col2': np.random.randn(num_rows)
})
df.to_csv('bench_data.csv', index=False)
print(f"Generated bench_data.csv with {num_rows} rows.")

## 4. Run Nairobi OS Fused Strike
Nairobi OS performs ingestion and analytics in a single fused pipeline.

In [ ]:
import json
import time
import psutil

# Allow daemon to settle
time.sleep(1)

start_time = time.time()
result_json = nairobi_os.data.pipeline("bench_data.csv", "target", "col1,col2")
end_time = time.time()

res = json.loads(result_json)
print(f"Nairobi OS Latency: {1000*(end_time - start_time):.2f}ms")
print(f"Correlation Result: {res['pearson']}")

# Stop Refinery
nairobi_os.stop_refinery()

## 5. Resource Comparison
Observe the Peak RSS usage in your Kaggle session dashboard.